# COMP5329 — Deep Learning

**Tutorial 10 — Deep Reinforcement Learning: From Q-Learning to LLM Alignment**

**Semester 1, 2026**

### Learning Objectives

By the end of this tutorial you will be able to:

1. Formulate a decision problem as a **Markov Decision Process** (MDP) and define policies, value functions, and Q-functions.
2. Derive the **Bellman equations** for state-value and action-value functions.
3. Implement **tabular Q-learning** from scratch on a grid world, including $\varepsilon$-greedy exploration.
4. Explain why tabular Q-learning fails in large state spaces and how **Deep Q-Networks (DQN)** overcome this with experience replay and target networks.
5. Implement a **DQN agent** from scratch and compare with tabular Q-learning.
6. Derive the **PPO clipped surrogate objective** and explain how RLHF uses a reward model + PPO to align LLMs.
7. Derive the **DPO loss function** from the RLHF objective via the closed-form optimal policy.
8. Explain **SLiC-HF**'s contrastive ranking approach and implement its max-margin loss.
9. Explain **GRPO**'s group-relative advantage estimation and how it eliminates the critic network.
10. Compare RLHF/PPO, DPO, SLiC-HF, and GRPO on complexity, components, stability, and use cases.

### Topic Coverage

Week 10 covers **deep reinforcement learning: from classical Q-learning to LLM alignment**. The full topic list (see `Week10_Self_Study_Deep_RL.ipynb`) is:

- ✅ **MDP framework** — states, actions, transitions, rewards, discount factor *(tutorial)*
- ✅ **Value functions & Bellman equations** — $V^\pi$, $Q^\pi$, optimality equations *(tutorial)*
- ✅ **Tabular Q-learning** — TD update, ε-greedy exploration, grid-world demo *(tutorial, with in-class practice)*
- ✅ **Deep Q-Networks (DQN)** — experience replay, target networks, neural Q-function *(tutorial)*
- 📖 **Historical breakthroughs** — Atari, AlphaGo/AlphaZero, StarCraft II *(self-study)*
- ✅ **RLHF with PPO** — clipped surrogate objective, the 4-model training loop *(tutorial)*
- ✅ **DPO** — closed-form optimal policy, log-sigmoid loss, derivation from RLHF *(tutorial, with in-class practice)*
- ✅ **SLiC-HF** — contrastive ranking without RL *(tutorial, briefly)*
- ✅ **GRPO** — group-relative advantages, no critic, verifiable-reward tasks *(tutorial, with in-class practice)*
- 📖 **InstructGPT case study** — empirical payoff of alignment *(self-study)*
- 📖 **Full unified comparison with citations and references** *(self-study)*

Due to time constraints, the tutorial compresses the full classical RL → alignment journey into 60 minutes. Historical breakthroughs, the InstructGPT case study, and fuller derivations are left as self-study — the self-study notebook covers them in depth across 18 chapters.

The live session is organised into three parts: **Part A** — tutor walkthrough, **Part B** — in-class coding exercises, **Part C** — exam-style Q&A.

> Sections 1–8 below are the **reference material** for Part A. The practice section and exam questions are at the bottom.

---
# Part A · Tutor Walkthrough

## 0. From Environment Rewards to Human Preferences

In Weeks 7-9 we built Transformers, scaled them to LLMs, and learned the alignment pipeline (pre-training $\to$ SFT $\to$ RLHF/DPO). Week 9 introduced the alignment formulas -- but WHERE does the "RL" in RLHF come from? What do "policy", "reward", and "optimisation" actually mean?

This tutorial has **two movements**:

**Movement 1: Classical RL** -- Build RL from first principles: MDP formulation, tabular Q-learning on a grid world, then Deep Q-Networks.

**Movement 2: RL for LLM Alignment** -- Connect RL to LLMs: RLHF/PPO (full pipeline), DPO (remove reward model), SLiC-HF (remove RL), GRPO (remove critic). Each method **progressively simplifies** the alignment pipeline.

| Section | Topic | Code Depth | Key Innovation |
|---|---|---|---|
| 1 | RL Foundations (MDP) | Pure math | Mathematical framework |
| 2 | Q-Learning | **From scratch** (grid world) | Tabular value iteration |
| 3 | Deep Q-Learning (DQN) | **From scratch** (neural net) | Function approximation |
| 4 | RLHF with PPO | Conceptual + PPO clip viz | RL for LLM alignment |
| 5 | DPO | Full derivation + loss code | Remove reward model |
| 6 | SLiC-HF | Light code (ranking loss) | Remove RL entirely |
| 7 | GRPO | Light code (group advantage) | Remove critic network |

---

## 1. RL Foundations: Markov Decision Processes

### 1.1 The MDP Framework

Reinforcement learning differs from supervised learning fundamentally: there are no labelled (input, output) pairs. Instead, an **agent** interacts with an **environment** by taking **actions**, receiving **rewards**, and observing **state transitions**.

Formally, an MDP is defined by $(S, A, P, R, \gamma)$:

| Symbol | Name | Description |
|---|---|---|
| $S$ | State space | All possible states |
| $A$ | Action space | All possible actions |
| $P(s' \mid s, a)$ | Transition function | Probability of reaching $s'$ from $s$ via $a$ |
| $R(s, a, s')$ | Reward function | Immediate reward for the transition |
| $\gamma \in [0, 1)$ | Discount factor | How much to value future vs. immediate rewards |

The **Markov property**: the next state depends only on the current state and action, not on the history.

### 1.2 Policy, Value Function, and Q-Function

A **policy** $\pi(a \mid s)$ maps states to a distribution over actions. The goal: find $\pi^*$ that maximises cumulative reward.

**State-value function** -- expected return from state $s$ under policy $\pi$:

$$V^\pi(s) = \mathbb{E}_\pi\left[\sum_{t=0}^{\infty} \gamma^t R_t \mid S_0 = s\right]$$

**Action-value function (Q-function)** -- expected return from taking $a$ in $s$, then following $\pi$:

$$Q^\pi(s, a) = \mathbb{E}_\pi\left[\sum_{t=0}^{\infty} \gamma^t R_t \mid S_0 = s, A_0 = a\right]$$

For the **optimal** policy: $V^*(s) = \max_a Q^*(s, a)$ and $\pi^*(s) = \arg\max_a Q^*(s, a)$.

### 1.3 The Bellman Equations

Value functions satisfy recursive relationships. The **Bellman optimality equation for $Q^*$**:

$$Q^*(s, a) = \mathbb{E}_{s'}\left[R(s, a, s') + \gamma \max_{a'} Q^*(s', a')\right]$$

This is the foundation of Q-learning: $Q^*$ is a **fixed point** of this equation, and we can iteratively converge to it.

**Connection to LLMs** (preview): In RLHF (Section 4), the LLM IS the policy $\pi_\theta$, the prompt is state $s$, the generated response is action $a$, and reward comes from a learned reward model $r_\phi(s, a)$.

---

## 2. Q-Learning: Tabular RL from Scratch

### 2.1 The Q-Learning Update Rule

Q-learning (Watkins, 1989) is an **off-policy** algorithm that learns $Q^*$ directly:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[\underbrace{r + \gamma \max_{a'} Q(s', a')}_{\text{TD target}} - Q(s, a)\right]$$

The term in brackets is the **temporal difference (TD) error**.

**$\varepsilon$-greedy exploration**: With probability $\varepsilon$, take a random action (explore); otherwise take $\arg\max_a Q(s, a)$ (exploit). $\varepsilon$ decays over time.

### 2.2 The Grid World

```
┌───┬───┬───┬───┬───┐
│ S │   │   │   │   │
├───┼───┼───┼───┼───┤
│   │ ▓ │   │ ▓ │   │
├───┼───┼───┼───┼───┤
│   │   │   │   │   │
├───┼───┼───┼───┼───┤
│   │ ▓ │   │ ▓ │   │
├───┼───┼───┼───┼───┤
│   │   │   │   │ ★ │
└───┴───┴───┴───┴───┘
S = Start (0,0)    ★ = Goal (+10)    ▓ = Wall
Step penalty: -0.1  (encourages shortest path)
```

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────────
import numpy as np
import random
import math
from collections import deque
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

%matplotlib inline

In [ ]:
# ── Grid World Environment ───────────────────────────────────────────────────

class GridWorld:
    """5x5 grid world with walls, start, and goal."""
    def __init__(self):
        self.size = 5
        self.start = (0, 0)
        self.goal = (4, 4)
        self.walls = {(1, 1), (1, 3), (3, 1), (3, 3)}
        self.actions = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # up, down, left, right
        self.action_names = ['↑', '↓', '←', '→']
        self.state = self.start

    def reset(self):
        self.state = self.start
        return self.state

    def step(self, action_idx):
        """Take action, return (next_state, reward, done)."""
        dr, dc = self.actions[action_idx]
        r, c = self.state
        nr, nc = r + dr, c + dc
        if 0 <= nr < self.size and 0 <= nc < self.size and (nr, nc) not in self.walls:
            self.state = (nr, nc)
        reward = -0.1  # step penalty
        done = (self.state == self.goal)
        if done:
            reward = 10.0
        return self.state, reward, done

    def state_to_idx(self, state):
        return state[0] * self.size + state[1]

In [ ]:
# ── Tabular Q-Learning ──────────────────────────────────────────────────────

def q_learning(env, episodes=500, alpha=0.1, gamma=0.99,
               epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995):
    """Train a Q-table using tabular Q-learning with epsilon-greedy."""
    n_states = env.size * env.size  # 25
    n_actions = len(env.actions)     # 4
    Q = np.zeros((n_states, n_actions))
    rewards_per_episode = []
    epsilon = epsilon_start

    for ep in range(episodes):
        state = env.reset()
        total_reward = 0
        done = False
        while not done:
            s_idx = env.state_to_idx(state)
            # Epsilon-greedy
            if np.random.random() < epsilon:
                action = np.random.randint(n_actions)
            else:
                action = np.argmax(Q[s_idx])
            next_state, reward, done = env.step(action)
            ns_idx = env.state_to_idx(next_state)
            # Q-learning update
            td_target = reward + gamma * np.max(Q[ns_idx]) * (1 - done)
            Q[s_idx, action] += alpha * (td_target - Q[s_idx, action])
            state = next_state
            total_reward += reward
        epsilon = max(epsilon_end, epsilon * epsilon_decay)
        rewards_per_episode.append(total_reward)
    return Q, rewards_per_episode

env = GridWorld()
Q_table, q_rewards = q_learning(env, episodes=500)
print(f'Average reward (last 50): {np.mean(q_rewards[-50:]):.2f}')

In [ ]:
# ── Q-value heatmap + learned policy ─────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))

# Left: V(s) heatmap
V = np.max(Q_table, axis=1).reshape(env.size, env.size)
im = ax1.imshow(V, cmap='RdYlGn', interpolation='nearest')
for (r, c) in env.walls:
    ax1.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, fill=True, color='gray'))
for r in range(env.size):
    for c in range(env.size):
        if (r, c) not in env.walls:
            ax1.text(c, r, f'{V[r,c]:.1f}', ha='center', va='center', fontsize=9)
ax1.set_title('Learned V(s) = max_a Q(s,a)', fontsize=12)
plt.colorbar(im, ax=ax1)

# Right: Policy arrows
ax2.set_xlim(-0.5, env.size-0.5); ax2.set_ylim(env.size-0.5, -0.5)
for r in range(env.size):
    for c in range(env.size):
        if (r, c) in env.walls:
            ax2.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, fill=True, color='gray'))
        elif (r, c) == env.goal:
            ax2.text(c, r, '★', ha='center', va='center', fontsize=20, color='gold')
        else:
            s_idx = env.state_to_idx((r, c))
            best = np.argmax(Q_table[s_idx])
            ax2.text(c, r, env.action_names[best], ha='center', va='center', fontsize=16)
ax2.set_title('Learned Optimal Policy π*(s)', fontsize=12)
ax2.set_aspect('equal'); ax2.grid(True, linewidth=0.5)
plt.tight_layout(); plt.show()

In [ ]:
# ── Training curve ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
w = 20
smoothed = np.convolve(q_rewards, np.ones(w)/w, mode='valid')
ax.plot(smoothed, color='steelblue', linewidth=1.5)
ax.set_xlabel('Episode'); ax.set_ylabel('Total Reward (smoothed)')
ax.set_title('Q-Learning Training Curve')
ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

> **Transition**: Tabular Q-learning works perfectly for our 25-state grid world. But what happens when the state space is enormous -- say, the raw pixels of an Atari game ($84 \times 84 \times 4 = 28{,}224$ dimensions)? We cannot maintain a Q-table with billions of entries. The solution: **approximate** $Q(s, a)$ with a neural network.

---

## 3. Deep Q-Networks (DQN)

### 3.1 From Table to Neural Network

Mnih et al. (2015) replaced the Q-table with a neural network $Q_\theta(s, a)$, trained to minimise:

$$\mathcal{L}(\theta) = \mathbb{E}_{(s,a,r,s') \sim \mathcal{D}} \left[\left(r + \gamma \max_{a'} Q_{\theta^-}(s', a') - Q_\theta(s, a)\right)^2\right]$$

Two key innovations:

1. **Experience Replay**: Store transitions in a buffer $\mathcal{D}$, sample random mini-batches. Breaks correlation between consecutive samples.
2. **Target Network**: A slowly-updated copy $Q_{\theta^-}$ computes the target. Prevents the target from moving with every gradient step.

Without either, training diverges.

In [ ]:
# ── Experience Replay + DQN Agent ─────────────────────────────────────────

class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)
    def push(self, s, a, r, s_next, done):
        self.buffer.append((s, a, r, s_next, done))
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, sn, d = zip(*batch)
        return (torch.FloatTensor(s), torch.LongTensor(a),
                torch.FloatTensor(r), torch.FloatTensor(sn), torch.FloatTensor(d))
    def __len__(self): return len(self.buffer)


class QNetwork(nn.Module):
    """MLP Q-network: state -> Q-values for each action."""
    def __init__(self, state_dim, n_actions, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions))
    def forward(self, x): return self.net(x)


class DQNAgent:
    """DQN with experience replay and target network."""
    def __init__(self, state_dim=2, n_actions=4, lr=1e-3, gamma=0.99,
                 batch_size=64, target_update=10):
        self.n_actions = n_actions
        self.gamma = gamma
        self.batch_size = batch_size
        self.target_update = target_update
        self.q_net = QNetwork(state_dim, n_actions)
        self.target_net = QNetwork(state_dim, n_actions)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.buffer = ReplayBuffer()
        self.steps = 0

    def select_action(self, state, epsilon):
        if random.random() < epsilon:
            return random.randint(0, self.n_actions - 1)
        with torch.no_grad():
            return self.q_net(torch.FloatTensor(state).unsqueeze(0)).argmax(1).item()

    def train_step(self):
        if len(self.buffer) < self.batch_size: return None
        s, a, r, sn, d = self.buffer.sample(self.batch_size)
        q_vals = self.q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            next_q = self.target_net(sn).max(1)[0]
            targets = r + self.gamma * next_q * (1 - d)
        loss = F.mse_loss(q_vals, targets)
        self.optimizer.zero_grad(); loss.backward(); self.optimizer.step()
        self.steps += 1
        if self.steps % self.target_update == 0:
            self.target_net.load_state_dict(self.q_net.state_dict())
        return loss.item()

In [ ]:
# ── Train DQN ──────────────────────────────────────────────────────────────
agent = DQNAgent(state_dim=2, n_actions=4)
env = GridWorld()
dqn_rewards = []
epsilon = 1.0

for ep in range(500):
    state = env.reset()
    total_reward = 0; done = False
    while not done:
        features = [state[0]/4.0, state[1]/4.0]  # normalise
        action = agent.select_action(features, epsilon)
        next_state, reward, done = env.step(action)
        agent.buffer.push(features, action, reward,
                          [next_state[0]/4.0, next_state[1]/4.0], float(done))
        agent.train_step()
        state = next_state; total_reward += reward
    epsilon = max(0.01, epsilon * 0.995)
    dqn_rewards.append(total_reward)
print(f'DQN average reward (last 50): {np.mean(dqn_rewards[-50:]):.2f}')

In [ ]:
# ── Q-Learning vs DQN comparison ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
w = 20
ax.plot(np.convolve(q_rewards, np.ones(w)/w, mode='valid'),
        label='Tabular Q-Learning', color='steelblue', linewidth=1.5)
ax.plot(np.convolve(dqn_rewards, np.ones(w)/w, mode='valid'),
        label='DQN', color='coral', linewidth=1.5)
ax.set_xlabel('Episode'); ax.set_ylabel('Total Reward (smoothed)')
ax.set_title('Tabular Q-Learning vs DQN on 5×5 Grid World')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

### 3.3 Tabular vs DQN

| Feature | Tabular Q-Learning | DQN |
|---|---|---|
| State representation | One entry per state | Neural network (any features) |
| Scalability | $O(|S| \times |A|)$ memory | Scales to high dimensions |
| Generalisation | None | Generalises across similar states |
| Key innovations | -- | Experience replay + target network |

> **Transition: From Environment Rewards to Human Preferences**
>
> In classical RL, the agent receives a numerical reward from the environment. But when we want an LLM to be "helpful, honest, and harmless", we cannot write a reward function -- these concepts are too nuanced. The solution: learn the reward from **human preferences**.
>
> | RL Concept | LLM Alignment |
> |---|---|
> | State $s$ | Prompt $x$ |
> | Action $a$ | Generated response $y$ |
> | Policy $\pi(a \mid s)$ | LLM $\pi_\theta(y \mid x)$ |
> | Reward $R(s, a)$ | Reward model $r_\phi(x, y)$ |
> | Value function $V(s)$ | Expected reward for a prompt |
>
> In LLM alignment, we typically treat the entire response as a single "action" (bandit setting).

---

## 4. RLHF with PPO

### 4.1 The RLHF Pipeline

**Phase 1: Reward Model** -- Collect preferences $y_w \succ y_l$ and train $r_\phi$ via the Bradley-Terry model:

$$\mathcal{L}_{\text{RM}} = -\mathbb{E}\left[\log \sigma\bigl(r_\phi(x, y_w) - r_\phi(x, y_l)\bigr)\right]$$

**Phase 2: RL Fine-Tuning** -- Maximise reward with KL penalty to prevent reward hacking:

$$\max_\theta \; \mathbb{E}_{x,\, y \sim \pi_\theta}\left[r_\phi(x, y)\right] - \beta\, D_{\text{KL}}\left[\pi_\theta \| \pi_{\text{ref}}\right]$$

### 4.2 PPO: The Clipped Surrogate Objective

PPO (Schulman et al., 2017) limits how much the policy changes per update. Define the probability ratio:

$$r_t(\theta) = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{\text{old}}}(a_t \mid s_t)}$$

The **clipped surrogate objective**:

$$\mathcal{L}^{\text{CLIP}}(\theta) = \mathbb{E}_t\left[\min\!\left(r_t(\theta)\, \hat{A}_t,\;\; \text{clip}(r_t(\theta), 1-\varepsilon, 1+\varepsilon)\, \hat{A}_t\right)\right]$$

- $\hat{A}_t$ is the **advantage estimate** (how much better this action is than average)
- $\varepsilon \approx 0.2$ is the clip parameter
- Positive advantage: increase probability, but cap at $1+\varepsilon$
- Negative advantage: decrease probability, but cap at $1-\varepsilon$

**Generalised Advantage Estimation (GAE)**:

$$\hat{A}_t^{\text{GAE}} = \sum_{l=0}^{\infty} (\gamma \lambda)^l \delta_{t+l}, \quad \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

In [ ]:
# ── PPO clipping mechanism visualisation ─────────────────────────────────

ratio = torch.linspace(0.0, 2.5, 500)
eps = 0.2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Positive advantage
A_pos = 1.0
unclipped = ratio * A_pos
clipped = torch.clamp(ratio, 1 - eps, 1 + eps) * A_pos
obj = torch.min(unclipped, clipped)
ax1.plot(ratio, unclipped, '--', color='gray', alpha=0.6, label='Unclipped')
ax1.plot(ratio, clipped, '--', color='orange', alpha=0.6, label='Clipped only')
ax1.plot(ratio, obj, color='steelblue', linewidth=2.5, label='PPO (min)')
ax1.axvline(x=1-eps, color='red', ls=':', alpha=0.5); ax1.axvline(x=1+eps, color='red', ls=':', alpha=0.5)
ax1.set_xlabel('Ratio r(θ)'); ax1.set_ylabel('Objective')
ax1.set_title('Â > 0 (good action): increase prob, but cap it'); ax1.legend(fontsize=9)

# Negative advantage
A_neg = -1.0
unclipped_n = ratio * A_neg
clipped_n = torch.clamp(ratio, 1 - eps, 1 + eps) * A_neg
obj_n = torch.min(unclipped_n, clipped_n)
ax2.plot(ratio, unclipped_n, '--', color='gray', alpha=0.6, label='Unclipped')
ax2.plot(ratio, clipped_n, '--', color='orange', alpha=0.6, label='Clipped only')
ax2.plot(ratio, obj_n, color='coral', linewidth=2.5, label='PPO (min)')
ax2.axvline(x=1-eps, color='red', ls=':', alpha=0.5); ax2.axvline(x=1+eps, color='red', ls=':', alpha=0.5)
ax2.set_xlabel('Ratio r(θ)'); ax2.set_ylabel('Objective')
ax2.set_title('Â < 0 (bad action): decrease prob, but cap it'); ax2.legend(fontsize=9)

plt.suptitle('PPO Clipped Surrogate Objective', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

### 4.3 Why RLHF is Complex

The full RLHF pipeline requires **four models simultaneously**:

1. **Policy model** $\pi_\theta$ (the LLM being trained)
2. **Reference model** $\pi_{\text{ref}}$ (frozen SFT model)
3. **Reward model** $r_\phi$ (trained on preference data)
4. **Value model** $V_\psi$ (critic, for advantage estimation)

This is extremely memory-intensive and introduces multiple instability sources.

**The next three algorithms each remove a component:**
- **DPO**: Remove reward model + value model (no RL)
- **SLiC-HF**: Remove RL entirely (ranking loss)
- **GRPO**: Remove value model (critic) via group-relative advantages

> **Transition**: Managing four models is burdensome. Rafailov et al. (2023) asked: what if the optimal policy under the RLHF objective has a **closed-form solution**? Then we could skip the RL loop entirely.

---

## 5. Direct Preference Optimization (DPO)

### 5.1 The Key Derivation (Deepening Week 9)

Start from the KL-constrained RLHF objective:

$$\max_\pi \; \mathbb{E}_{x,y \sim \pi}\left[r(x,y) - \beta \log \frac{\pi(y|x)}{\pi_{\text{ref}}(y|x)}\right]$$

**Step 1** -- The optimal policy has a closed form:

$$\pi^*(y|x) = \frac{1}{Z(x)}\, \pi_{\text{ref}}(y|x)\, \exp\!\left(\frac{r(x,y)}{\beta}\right)$$

**Step 2** -- Reparameterise the reward in terms of the policy:

$$r(x,y) = \beta \log \frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)} + \beta \log Z(x)$$

**Step 3** -- Substitute into the Bradley-Terry preference model. The $Z(x)$ terms cancel!

$$\mathcal{L}_{\text{DPO}} = -\mathbb{E}_{(x,y_w,y_l)}\left[\log \sigma\!\left(\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)\right]$$

**Why DPO is elegant**: No reward model, no PPO instability, no online generation. Just a classification-like loss on preference pairs.

### 5.2 What $\beta$ Controls

- $\beta \to 0$: The policy can deviate arbitrarily from $\pi_{\text{ref}}$ -- no regularisation, may overfit to preferences.
- $\beta \to \infty$: The policy stays very close to $\pi_{\text{ref}}$ -- barely learns from preferences.
- Typical: $\beta = 0.1$ balances preference learning and regularisation.

In [ ]:
# ── DPO loss ────────────────────────────────────────────────────────────────

def dpo_loss(pi_w, pi_l, ref_w, ref_l, beta=0.1):
    """DPO loss. All inputs are log-probabilities, shape (B,)."""
    logits = beta * ((pi_w - ref_w) - (pi_l - ref_l))
    return -F.logsigmoid(logits).mean()

# Verification
pi_w  = torch.tensor([-1.0, -1.5])   # policy prefers y_w (higher log-prob)
pi_l  = torch.tensor([-3.0, -4.0])
ref_w = torch.tensor([-2.0, -2.0])
ref_l = torch.tensor([-2.0, -2.0])
print(f'Loss (correct preference):   {dpo_loss(pi_w, pi_l, ref_w, ref_l):.4f}')
print(f'Loss (incorrect preference): {dpo_loss(pi_l, pi_w, ref_w, ref_l):.4f}')
print('Correct preference gives LOWER loss.')

---

> **SLiC-HF and GRPO — see the Self-Study material.**
>
> DPO removes the reward model by exploiting the closed-form optimum. Two further simplifications deserve a mention before the comparison table:
>
> - **SLiC-HF** (Zhao et al., 2023) drops the reference model entirely and uses a max-margin ranking loss on preference pairs.
> - **GRPO** (DeepSeek, 2024) keeps PPO's clipped objective but replaces the value network with group-relative reward statistics — zero extra parameters.
>
> Full derivations, gradient analyses, and the DeepSeek-R1 case study live in **`Week10_Self_Study_Deep_RL.ipynb`**. The one-line summary for each method is in Section 8 below, and you will implement a GRPO update step yourself in Practice 3.


---

## 8. Grand Comparison

### 8.1 LLM Alignment Methods

| Feature | RLHF / PPO | DPO | SLiC-HF | GRPO |
|---|---|---|---|---|
| **Reward model** | Required | No | No | No (verifier) |
| **Critic (value net)** | Required | No | No | **No** |
| **Reference model** | Required | Required | No | Required |
| **Online sampling** | Yes | No | No | **Yes** |
| **Loss type** | PPO clipped surrogate | Log-sigmoid | Max-margin | PPO + group advantages |
| **Training complexity** | Very high (4 models) | Low (2 models) | Lowest (1 model) | Medium (2 models) |
| **Best for** | General alignment | General alignment | Offline preferences | Verifiable tasks |

### 8.2 The Simplification Trajectory

```
RLHF/PPO:  Policy + Reference + Reward Model + Value Model  (4 models)
DPO:       Policy + Reference                                (2 models, remove RM + Value)
SLiC-HF:   Policy alone                                      (1 model, remove RL + Ref)
GRPO:      Policy + Reference (no Critic, use group stats)   (2 models, remove Value)
```

### 8.3 Full RL Evolution (Part A + Part B)

```
Part A:  Tabular Q-Learning → DQN (neural Q-function)
              │
         "environment rewards → human preferences"
              │
Part B:  RLHF/PPO (full RL) → DPO (no RL) → SLiC-HF (ranking) → GRPO (no critic)
```

---
# Part B · In-Class Coding Exercises

Complete the three exercises below. Each one has `TODO` blanks to fill in. After working through them your tutor will walk through the solutions.

The three exercises mirror the three most important takeaways of this tutorial:

1. **Tabular Q-learning** (classical RL core) — the TD update + ε-greedy exploration.
2. **DPO loss** (LLM alignment core) — translating the closed-form derivation into ~1 line of PyTorch.
3. **GRPO step** (latest alignment method) — group-relative advantages + PPO-clipped objective.

### Practice 1 — Tabular Q-learning on a 4-state chain

We use a tiny 4-state chain so you can focus on the **update rule**, not the environment:

```
[S0] ── [S1] ── [S2] ── [G]        actions: 0 = left, 1 = right
                                    step reward = -0.1, goal reward = +1
```

**Your tasks** (in `train_chain` below):
- **TODO 1**: ε-greedy action selection.
- **TODO 2**: The Q-learning TD target and update.

After you finish, run the follow-up cell and explain **why `gamma=0.0` breaks learning**.


In [ ]:
# ── Practice 1: chain environment + Q-learning (TODO) ───────────────────────

class ChainEnv:
    def __init__(self, n=4):
        self.n = n
    def reset(self):
        self.state = 0
        return self.state
    def step(self, a):  # 0 = left, 1 = right
        if a == 1:
            self.state = min(self.state + 1, self.n - 1)
        else:
            self.state = max(self.state - 1, 0)
        done = (self.state == self.n - 1)
        reward = 1.0 if done else -0.1
        return self.state, reward, done


def train_chain(gamma=0.9, episodes=300, alpha=0.5, eps=0.2, seed=0):
    np.random.seed(seed)
    env = ChainEnv()
    Q = np.zeros((env.n, 2))
    for _ in range(episodes):
        s = env.reset()
        done = False
        while not done:
            # ── TODO 1: ε-greedy action selection ────────────────────────────
            # With probability `eps`, pick a random action in {0, 1};
            # otherwise pick argmax over Q[s].
            a = None  # <-- replace

            s_next, r, done = env.step(a)

            # ── TODO 2: Q-learning TD target and update ──────────────────────
            # TD target = r + gamma * max_{a'} Q[s_next, a']   (zero after terminal)
            # Then:  Q[s, a] += alpha * (td_target - Q[s, a])
            td_target = None  # <-- replace
            # Q[s, a] += ...   # <-- replace

            s = s_next
    return Q


In [ ]:
# ── Practice 1: solution ────────────────────────────────────────────────────

def train_chain_sol(gamma=0.9, episodes=300, alpha=0.5, eps=0.2, seed=0):
    np.random.seed(seed)
    env = ChainEnv()
    Q = np.zeros((env.n, 2))
    for _ in range(episodes):
        s = env.reset()
        done = False
        while not done:
            if np.random.random() < eps:
                a = np.random.randint(2)
            else:
                a = int(np.argmax(Q[s]))
            s_next, r, done = env.step(a)
            td_target = r + gamma * np.max(Q[s_next]) * (1.0 - float(done))
            Q[s, a] += alpha * (td_target - Q[s, a])
            s = s_next
    return Q


Q_good = train_chain_sol(gamma=0.9)
Q_bad  = train_chain_sol(gamma=0.0)
print('Q-table with gamma = 0.9 (rows = states S0..G, cols = [left, right]):')
print(np.round(Q_good, 3))
print('\nQ-table with gamma = 0.0:')
print(np.round(Q_bad, 3))

# Discussion:
# With gamma = 0.0, the TD target collapses to just `r`, so every non-terminal
# step only sees the -0.1 step penalty. The +1 goal reward NEVER propagates
# backwards to earlier states -> the agent has no signal that "right" is
# the path to the goal, and Q-values for non-terminal states stay slightly
# negative with no preference between left and right.


### Practice 2 — DPO loss from the formula

The DPO loss (Section 5) is:

$$\mathcal{L}_{\text{DPO}} = -\log \sigma\!\left(\beta \bigl[(\log\pi_\theta(y_w|x) - \log\pi_{\text{ref}}(y_w|x)) - (\log\pi_\theta(y_l|x) - \log\pi_{\text{ref}}(y_l|x))\bigr]\right)$$

**TODO**: implement it below.

**Follow-up questions** (think before you run the cell):
1. If $\pi_\theta$ is exactly equal to $\pi_{\text{ref}}$ at initialisation, what is the loss value?
2. What is $\partial \mathcal{L}_{\text{DPO}} / \partial \log\pi_\theta(y_w|x)$ at that point? (Hint: derivative of $-\log\sigma(z)$ is $-\sigma(-z)$.)


In [ ]:
# ── Practice 2: DPO loss (TODO) ─────────────────────────────────────────────

def dpo_loss_practice(logp_w, logp_l, ref_w, ref_l, beta=0.1):
    """All four inputs are log-probs of shape (B,). Return scalar loss."""
    # ── TODO: build the logits inside sigmoid, then return -logsigmoid(logits).mean()
    logits = None  # <-- replace
    return -F.logsigmoid(logits).mean()


# Quick test harness (uncomment once TODO is filled in):
# logp_w = torch.tensor([-1.0, -1.5])
# logp_l = torch.tensor([-3.0, -4.0])
# ref_w  = torch.tensor([-2.0, -2.0])
# ref_l  = torch.tensor([-2.0, -2.0])
# print('loss (correct preference): ', dpo_loss_practice(logp_w, logp_l, ref_w, ref_l).item())
# print('loss (swapped preference): ', dpo_loss_practice(logp_l, logp_w, ref_w, ref_l).item())


In [ ]:
# ── Practice 2: solution + answers to follow-up questions ───────────────────

def dpo_loss_sol(logp_w, logp_l, ref_w, ref_l, beta=0.1):
    logits = beta * ((logp_w - ref_w) - (logp_l - ref_l))
    return -F.logsigmoid(logits).mean()


# Q1: If pi_theta == pi_ref, then logp_w == ref_w and logp_l == ref_l,
#     so logits = 0 and loss = -log(sigmoid(0)) = -log(0.5) = log(2) ≈ 0.6931.
logp_w = torch.tensor([-1.0]); logp_l = torch.tensor([-3.0])
ref_w  = torch.tensor([-1.0]); ref_l  = torch.tensor([-3.0])  # policy == reference
print('Loss when pi_theta == pi_ref:', dpo_loss_sol(logp_w, logp_l, ref_w, ref_l).item())
print('log(2) =', float(torch.log(torch.tensor(2.0))))

# Q2: d/d(logp_w) of -log sigma(z)  where z = beta*((logp_w - ref_w) - (logp_l - ref_l))
#     = -sigma(-z) * beta.
#     At initialisation z = 0, so the gradient is -beta * 0.5.
#     Meaning: a small uniform positive push on logp_w(preferred).  This is
#     why DPO training starts moving immediately even when the policy is
#     identical to the reference — unlike SLiC-HF, whose max-margin loss is
#     zero whenever the margin is already satisfied.


### Practice 3 — One GRPO step

You are given, for a single prompt $x$, a **group** of $G=8$ sampled responses and three tensors:

- `rewards` — verifiable reward for each sample (1 = math answer correct, 0 = wrong).
- `old_logp` — log-probability of each sample under $\pi_{\theta_{\text{old}}}$ (the sampling policy).
- `new_logp` — log-probability of the same samples under the current $\pi_\theta$ we are training.

Implement one GRPO update step:

1. **Group-relative advantages**: $\hat A_i = (r_i - \bar r) / (\sigma_r + \epsilon)$.
2. **Probability ratio**: $\rho_i = \exp(\log\pi_\theta - \log\pi_{\text{old}})$.
3. **PPO-clipped surrogate**: $\mathcal{L} = -\frac{1}{G}\sum_i \min(\rho_i \hat A_i,\; \text{clip}(\rho_i, 1-\varepsilon, 1+\varepsilon)\,\hat A_i)$.


In [ ]:
# ── Practice 3: GRPO step (TODO) ───────────────────────────────────────────

def grpo_step(rewards, old_logp, new_logp, eps_clip=0.2):
    """All inputs shape (G,). Return scalar loss for one group."""
    # ── TODO 1: group-relative advantages (normalise rewards within the group)
    advantages = None  # <-- replace

    # ── TODO 2: probability ratio pi_theta / pi_old
    ratio = None  # <-- replace

    # ── TODO 3: PPO-clipped surrogate objective → negate to get a loss
    unclipped = None  # <-- replace
    clipped   = None  # <-- replace
    loss = None  # <-- replace   (hint: -(torch.min(...).mean()))
    return loss


# Example group: 8 responses, 3 correct; small log-prob drift after one update.
# rewards  = torch.tensor([1., 0., 0., 1., 0., 0., 0., 1.])
# old_logp = torch.tensor([-5.0, -4.0, -6.0, -4.5, -5.5, -5.0, -6.5, -4.0])
# new_logp = old_logp + torch.tensor([0.1, -0.05, -0.1, 0.15, -0.05, -0.1, -0.1, 0.2])
# print('GRPO loss:', grpo_step(rewards, old_logp, new_logp).item())


In [ ]:
# ── Practice 3: solution ───────────────────────────────────────────────────

def grpo_step_sol(rewards, old_logp, new_logp, eps_clip=0.2):
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-8)
    ratio = torch.exp(new_logp - old_logp)
    unclipped = ratio * advantages
    clipped   = torch.clamp(ratio, 1 - eps_clip, 1 + eps_clip) * advantages
    loss = -torch.min(unclipped, clipped).mean()
    return loss


rewards  = torch.tensor([1., 0., 0., 1., 0., 0., 0., 1.])
old_logp = torch.tensor([-5.0, -4.0, -6.0, -4.5, -5.5, -5.0, -6.5, -4.0])
new_logp = old_logp + torch.tensor([0.1, -0.05, -0.1, 0.15, -0.05, -0.1, -0.1, 0.2])

adv = (rewards - rewards.mean()) / (rewards.std() + 1e-8)
print('rewards   :', rewards.tolist())
print('advantages:', [round(x, 3) for x in adv.tolist()])
print('GRPO loss :', grpo_step_sol(rewards, old_logp, new_logp).item())
print()
print('Note how the 3 correct responses all received POSITIVE advantage and')
print('the 5 incorrect ones NEGATIVE — all without any value network.')


---
# Part C · Exam-Style Questions

Three medium–hard short-answer questions. Attempt them first, then we will discuss the solutions together. They integrate content from earlier weeks (Week 6 backprop, Week 9 Bradley–Terry / RLHF). Answer sketches are provided in collapsed cells directly below each question — expand them only after attempting.

### Q1 — Bellman update + linear function approximation  *(integrates Week 6 backprop)*

Consider the following 3-state MDP with two actions `L, R`, deterministic transitions, and discount $\gamma = 0.9$.

| State | Action `L` → | Action `R` → | Reward on arrival |
|---|---|---|---|
| $s_0$ | stay at $s_0$ | go to $s_1$ | 0 |
| $s_1$ | go to $s_0$ | go to $s_2$ | 0 |
| $s_2$ | — (terminal) | — (terminal) | +1 when you arrive |

**(a)** Compute $Q^*(s, a)$ for every state–action pair by hand using the Bellman optimality equation.

**(b)** You are now told to replace the Q-table with a linear function approximator
$$\hat Q_\theta(s, a) = \theta^\top \phi(s, a),$$
where $\phi(s, a) \in \mathbb{R}^d$ is a fixed feature vector. Write down the squared TD loss for a single transition $(s, a, r, s')$ and derive $\partial \mathcal{L} / \partial \theta$ explicitly (remember: the target is treated as a constant, mirroring the "target network" trick in DQN).

**(c)** Explain in 1–2 sentences why treating the target as constant is important — what would go wrong if you also back-propagated through the $\max_{a'} \hat Q_\theta(s', a')$ term?

<details><summary><b>▸ Answer sketch — Q1</b></summary>

**(a)** Work backwards from the terminal state. $s_2$ is terminal, so $V^*(s_2) = 0$.

- $Q^*(s_1, R) = r(\text{arrive } s_2) + \gamma V^*(s_2) = 1 + 0.9 \cdot 0 = \mathbf{1.0}$.
- $V^*(s_1) = \max(Q^*(s_1, L), Q^*(s_1, R))$. We will use this below.
- $Q^*(s_0, R) = 0 + 0.9 \cdot V^*(s_1) = 0.9 \cdot 1.0 = \mathbf{0.9}$ (using $V^*(s_1) = 1$, verified below).
- $Q^*(s_0, L) = 0 + 0.9 \cdot V^*(s_0)$, a self-loop. Setting $V^*(s_0) = \max(Q^*(s_0, L), 0.9)$: if the max is $0.9$, then $Q^*(s_0, L) = 0.9 \cdot 0.9 = \mathbf{0.81}$, and indeed $\max(0.81, 0.9) = 0.9$ — self-consistent.
- $Q^*(s_1, L) = 0 + 0.9 \cdot V^*(s_0) = 0.9 \cdot 0.9 = \mathbf{0.81}$.
- Check $V^*(s_1) = \max(0.81, 1.0) = 1.0$ ✓.

Summary: $Q^* = \{(s_0, L) : 0.81,\ (s_0, R) : 0.9,\ (s_1, L) : 0.81,\ (s_1, R) : 1.0\}$, and the optimal policy is "always R".

**(b)** For one transition $(s, a, r, s')$, the squared TD loss is
$$\mathcal{L}(\theta) = \tfrac{1}{2}\left(y - \hat Q_\theta(s, a)\right)^2, \qquad y = r + \gamma \max_{a'} \hat Q_{\theta^-}\!(s', a'),$$
where $\theta^-$ is a **stopped-gradient** copy of $\theta$ (the target network). Since $\hat Q_\theta(s, a) = \theta^\top \phi(s, a)$,
$$\frac{\partial \mathcal{L}}{\partial \theta} = -\left(y - \theta^\top \phi(s, a)\right) \phi(s, a).$$
This is precisely a gradient-descent step on a linear regression whose label is $y$ — the standard semi-gradient Q-learning update.

**(c)** If you also back-propagate through $\max_{a'} \hat Q_\theta(s', a')$, the target itself becomes a function of $\theta$. Each gradient step then shifts *both* the prediction and the target it is chasing, which turns the update into a **moving-target** chase and breaks the contraction argument that guarantees Q-learning's convergence. Empirically, this causes the Bellman residual to oscillate and the network to diverge — this is exactly the pathology the target-network trick in DQN (or, equivalently, the `.detach()` in modern code) exists to prevent.
</details>

### Q2 — Deriving DPO from RLHF  *(integrates Week 9 Bradley–Terry)*

Start from the KL-constrained RLHF objective for a single prompt $x$:

$$\max_\pi \; \mathbb{E}_{y \sim \pi(\cdot|x)}\bigl[r(x, y)\bigr] \; - \; \beta\, D_{\text{KL}}\!\bigl[\pi(\cdot|x) \,\|\, \pi_{\text{ref}}(\cdot|x)\bigr].$$

**(a)** Show that the optimal policy has the closed form
$$\pi^*(y|x) = \frac{1}{Z(x)}\, \pi_{\text{ref}}(y|x) \exp\!\left(\frac{r(x,y)}{\beta}\right),$$
and state clearly what $Z(x)$ is.

**(b)** Use (a) to re-parameterise the reward $r(x, y)$ in terms of $\pi^*$, $\pi_{\text{ref}}$, and $Z(x)$.

**(c)** Substitute your expression for $r$ into the Bradley–Terry log-likelihood
$$\log \sigma\!\bigl(r(x, y_w) - r(x, y_l)\bigr)$$
and show explicitly that $Z(x)$ cancels, giving the DPO loss.

**(d)** One sentence: why does DPO not need a reward model at inference time?

<details><summary><b>▸ Answer sketch — Q2</b></summary>

**(a)** Expand the KL to get the Lagrangian
$$J(\pi) = \sum_y \pi(y|x)\,r(x,y) - \beta \sum_y \pi(y|x) \log \frac{\pi(y|x)}{\pi_{\text{ref}}(y|x)} - \lambda\!\left(\sum_y \pi(y|x) - 1\right).$$
Differentiating w.r.t. $\pi(y|x)$ and setting to zero gives
$$r(x,y) - \beta\!\left(\log \frac{\pi(y|x)}{\pi_{\text{ref}}(y|x)} + 1\right) - \lambda = 0,$$
so $\pi^*(y|x) \propto \pi_{\text{ref}}(y|x) \exp(r(x,y)/\beta)$. Normalising gives
$$\pi^*(y|x) = \frac{1}{Z(x)}\, \pi_{\text{ref}}(y|x) \exp(r(x,y)/\beta), \qquad Z(x) = \sum_{y'} \pi_{\text{ref}}(y'|x)\exp(r(x,y')/\beta).$$
$Z(x)$ is the **partition function**: a normaliser that depends only on the prompt $x$, not on any particular response $y$.

**(b)** Taking logs of the closed form and solving for $r$:
$$r(x, y) = \beta \log \frac{\pi^*(y|x)}{\pi_{\text{ref}}(y|x)} + \beta \log Z(x).$$
The first term is the log-ratio of the optimal policy to the reference; the second is a prompt-dependent constant.

**(c)** Substituting both $r(x, y_w)$ and $r(x, y_l)$ into the Bradley–Terry preference log-likelihood:
$$r(x, y_w) - r(x, y_l) = \beta\left[\log \frac{\pi^*(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \log \frac{\pi^*(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right] + \beta\bigl[\log Z(x) - \log Z(x)\bigr].$$
The $\log Z(x)$ terms **cancel exactly** because they depend only on $x$ and not on $y$. Taking the negative log of $\sigma(\cdot)$ and replacing $\pi^*$ with the trainable $\pi_\theta$ gives the DPO loss:
$$\mathcal{L}_{\text{DPO}}(\theta) = -\log \sigma\!\left(\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right).$$

**(d)** Because DPO's loss already expresses the reward *implicitly* as the log-ratio $\beta\log(\pi_\theta/\pi_{\text{ref}})$, the trained $\pi_\theta$ itself encodes everything we need — at inference we just sample from $\pi_\theta$ and never instantiate a reward model. The reward model has been absorbed into the policy's parameters.
</details>

### Q3 — Method selection + memory accounting

A team wants to fine-tune a **7B-parameter** LLM for **mathematical reasoning**, where every answer can be programmatically verified against a ground-truth solution. They want to compare four alignment methods: **RLHF/PPO, DPO, SLiC-HF, GRPO**.

**(a)** For **RLHF/PPO**, list every model that must be held in memory during training and give a rough estimate (order of magnitude, in GB) of the total parameter memory in `fp16`. Assume each 7B model ≈ 14 GB in fp16.

**(b)** For each of **DPO**, **SLiC-HF**, and **GRPO**, state which model(s) from (a) are *removed*, and briefly justify why the method can afford to remove them (1 sentence each).

**(c)** Given the team's verifiable-reward setting, recommend one method and give two concrete reasons why it is better suited than the other three for this specific use case.

**(d)** Describe **one potential failure mode** of your recommended method. (Think: what happens if every sampled response in a group has the same reward? What does that do to the advantages?)

<details><summary><b>▸ Answer sketch — Q3</b></summary>

**(a)** RLHF/PPO holds **four** models simultaneously during training:
1. **Policy** $\pi_\theta$ — being trained (14 GB parameters + gradients + optimiser state, so effectively ~56 GB with Adam).
2. **Reference policy** $\pi_{\text{ref}}$ — frozen, forward only (14 GB).
3. **Reward model** $r_\phi$ — frozen at RL time but trained in an earlier phase; forward only during PPO (14 GB).
4. **Value/critic** $V_\psi$ — trained alongside the policy (14 GB + gradients).

Parameter memory alone is roughly $4 \times 14 = 56$ GB; counting optimiser state for the trainable ones pushes this well over $100$ GB, which is why RLHF at 7B already needs a multi-GPU setup.

**(b)**
- **DPO** removes the **reward model** and the **value/critic**. The implicit-reward re-parameterisation (Q2) makes the explicit $r_\phi$ unnecessary, and because DPO is a purely supervised loss on preference pairs there is no RL loop and no baseline/value network is needed.
- **SLiC-HF** removes the **reward model, the value/critic, and the reference model**. Its max-margin ranking loss operates directly on two responses without needing a baseline or a KL anchor — but the price is an additional hyperparameter and a slightly different objective that no longer connects analytically to RLHF.
- **GRPO** removes only the **value/critic**. It replaces the learned value baseline with a **group-relative** one: for each prompt, sample $G$ responses, and use $(r_i - \bar r)/\sigma_r$ as the advantage. It still needs the reference (KL penalty, like PPO) and a reward source — but in the verifiable-reward case the "reward model" is replaced by a cheap rule-based verifier.

**(c)** **Recommendation: GRPO.** Two concrete reasons:

1. **Verifier replaces the reward model.** Because every math answer can be programmatically checked, the "reward" is a binary correct/incorrect signal produced instantly and for free — no preference data collection, no RM training, no RM memory cost at RL time. GRPO is designed exactly for this regime.
2. **Group-relative advantages exploit multi-sample generation cheaply.** For reasoning tasks we already want to sample many rollouts per prompt (e.g., for self-consistency, majority voting). GRPO reuses those samples to compute advantages without any learned critic, saving another ~14 GB of parameters and avoiding the notoriously brittle critic training. DPO cannot use the verifier signal this directly, SLiC-HF is offline and wastes the dynamic rollout structure, and full RLHF/PPO pays for a RM and a critic that we do not need here.

**(d)** When **every sample in a group has the same reward** (e.g. all 8 rollouts are correct or all 8 are wrong), the group mean equals every sample's reward, so the advantage $\hat A_i = (r_i - \bar r)/\sigma_r$ is **zero for every sample** and the gradient on that prompt vanishes — no learning signal whatsoever. This is particularly bad for prompts that are either too easy (always correct) or too hard (always wrong): exactly the prompts at the boundary of the model's capability — the ones we most want to train on — are the only ones producing signal. Mitigations include (i) filtering out degenerate groups before the update, (ii) increasing group size $G$ so at least one success/failure is likely, and (iii) curriculum-learning the prompt difficulty to keep most groups mixed.
</details>